# Wrongway - self-play training of the deployed agent on Kaggle

Runs `training/alphazero/selfplay_train.py`: the agent's own search (alpha-beta + net at the root,
the code `az_agent.py` plays) plays itself, 30% of games against the no-network engine (= MSSV).
Each generation: self-play -> train -> gate (vs best, vs MSSV; scores split moving first / second)
-> accepted nets embedded into `export/az_agent_sp.py`. Starts from the `std1` net, no bootstrap.
The run is mirrored to the Hugging Face repo after the start and after every generation.

**Before running**
- The branch in `BRANCH` must be pushed to GitHub (the notebook clones it).
- Settings: Accelerator **GPU T4 x2**, Internet **on**.
- Add-ons -> Secrets: `HF_STORAGE_TOKEN` (HF write token) and `ON_KAGGLE`.
- Use **Save Version -> Save & Run All (Commit)** so it keeps running after you close the tab.

**Time.** Self-play and gate games run on the CPU (Kaggle: 4 cores -> 3 game workers); only the
training step uses the GPU. Rough guess: ~30 min per generation at the default settings. The
script pauses (exit code 3) before a generation that would cross `STOP_AFTER_HOURS`; commit the
notebook again and it pulls the run from Hugging Face and continues.

## 1. Hugging Face login

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
onkaggle = bool(user_secrets.get_secret("ON_KAGGLE"))
hftoken = user_secrets.get_secret("HF_STORAGE_TOKEN")

if hftoken:
    login(hftoken)
else:
    print("Error: HF_TOKEN (HUGGINGFACE TOKEN) environment variable not set.")

## 2. Settings

In [ ]:
REPO_URL = "https://github.com/edamleanh/wrongway.git"
BRANCH = "feat/selfplay-train"
RUN = "sp1"                         # run name; keep it the same across sessions to resume
HF_REPO = "henrytang785/az_agent"   # private model repo (created if missing)
STOP_AFTER_HOURS = 11.0             # Kaggle kills the session at 12 h
GENERATIONS = 20                    # total generations for the run (raise later to continue)

# Starting net (new run only). Default: full-precision std1 checkpoint from HF (policy + trained
# value head). "" -> MODEL_BLOB of submissions/az_agent_std1.py from the repo (float16, value head fresh).
INIT_HF_FILE = "best/std1/best.pt"

# Extra config overrides, e.g. ["sp_budget=0.3", "sp_games=150", "league_frac=0.5"] (fields: SPConfig).
EXTRA_SET = []

WORKDIR = "/kaggle/working/wrongway"
RUNS_DIR = "/tmp/az_runs"            # big data shards stay out of the saved notebook output (HF keeps them)

## 3. Clone the repo and install dependencies

In [ ]:
import os, subprocess

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "log", "--oneline", "-1"], check=True)
os.makedirs(RUNS_DIR, exist_ok=True)
os.chdir(WORKDIR)

In [ ]:
# torch + CUDA are preinstalled on Kaggle; only add what the pipeline needs on top.
!pip install -q "huggingface_hub>=0.23" tensorboard pytest

## 4. Hardware check

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  ", i, torch.cuda.get_device_name(i))
print("CPU cores:", os.cpu_count(), "-> game workers:", max(1, os.cpu_count() - 1))

## 5. Resume from Hugging Face, or fetch the starting net

Downloads `runs/<RUN>/` if an earlier session pushed it. First session: downloads `INIT_HF_FILE`
(falls back to `az_agent_std1.py` from the repo if that file is missing).

In [ ]:
import sys
sys.path.insert(0, WORKDIR)
os.environ["AZ_RUNS_DIR"] = RUNS_DIR
from training.alphazero.hf_sync import pull

resumed = pull(RUN, HF_REPO, RUNS_DIR, marker="selfplay.json")
init = os.path.join(WORKDIR, "wrongway-student/wrongway/submissions/az_agent_std1.py")
if not resumed and INIT_HF_FILE:
    from huggingface_hub import hf_hub_download
    try:
        init = hf_hub_download(HF_REPO, INIT_HF_FILE, local_dir="/tmp/az_init")
    except Exception as exc:
        print("could not download %s (%s) - using %s" % (INIT_HF_FILE, exc, init))
print("resume" if resumed else "new run from " + init)

## 6. Train

Runs `selfplay_train.py` in a subprocess and streams the log. A resumed run continues at the next
generation. Exit code 3 = paused at the time limit: commit this notebook again to continue.

In [ ]:
state = os.path.join(RUNS_DIR, RUN, "selfplay.json")
cmd = [sys.executable, "-u", "-m", "training.alphazero.selfplay_train", "--run", RUN,
       "--set", "stop_after_hours=%s" % STOP_AFTER_HOURS, "--set", "generations=%d" % GENERATIONS,
       "--set", "data_parallel=true"]
for item in EXTRA_SET:
    cmd += ["--set", item]
if not os.path.exists(state):
    cmd += ["--init", init]
env = dict(os.environ, AZ_RUNS_DIR=RUNS_DIR, AZ_HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(" ".join(cmd))

proc = subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(os.path.join(RUNS_DIR, RUN + ".log"), "a") as log:
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
exit_code = proc.wait()
if exit_code == 3:
    print("\nPAUSED at the time limit - everything is on Hugging Face. Commit this notebook again to continue.")
elif exit_code:
    print("\nFAILED with exit code", exit_code, "- see the log above")
else:
    print("\nALL GENERATIONS DONE")

## 7. Status and files on Hugging Face

In [ ]:
subprocess.run([sys.executable, "-m", "training.alphazero.selfplay_train", "--run", RUN, "--status"],
               cwd=WORKDIR, env=env)
from huggingface_hub import list_repo_files
best = [f for f in list_repo_files(HF_REPO) if f.startswith("best/" + RUN + "/")]
print("\n".join(best))
print("https://huggingface.co/" + HF_REPO)

## 8. Final push (safety net)

The script already pushes after the start and after every generation; this catches anything
written after the last automatic push (for example if a push failed on a network error).

In [ ]:
subprocess.run([sys.executable, "-m", "training.alphazero.hf_sync", "push", "--run", RUN, "--repo", HF_REPO],
               cwd=WORKDIR, env=env)

## 9. Use the result on your machine

```bash
hf download henrytang785/az_agent best/sp1/az_agent_sp.py --local-dir az_hf
python training/gauntlet.py az_hf/best/sp1/az_agent_sp.py MSSV --pairs 40 --workers 6
hf download henrytang785/az_agent --include "runs/sp1/tb/*" --local-dir az_hf
uv run tensorboard --logdir az_hf/runs/sp1/tb
```